# ExoAstro: Baseline Model Training

This notebook demonstrates the end-to-end pipeline for training the baseline Advanced AstroNet model using the curated dataset. It loads the light curve views, normalizes them, and trains the model with gradient clipping and early stopping.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import callbacks
import matplotlib.pyplot as plt

# Ensure src modules can be imported
sys.path.append(os.path.abspath('..'))

from src.models.model import AdvancedAstroNetModel

## 1. Load the Curated Dataset
We load a sample mapping file that contains `target_id` and `label` (0: Transit, 1: Eclipsing Binary, 2: Blend, 3: Noise).

In [ ]:
def load_curated_dataset(data_dir, mapping_csv):
    df = pd.read_csv(mapping_csv)
    global_views, local_views, labels = [], [], []
    
    for idx, row in df.iterrows():
        target_id = str(row['target_id'] if 'target_id' in row else row.iloc[0])
        label = int(row['label'] if 'label' in row else row.iloc[1])
        
        path = os.path.join(data_dir, target_id)
        g_path = os.path.join(path, "global_view.npy")
        l_path = os.path.join(path, "local_view.npy")
        
        if os.path.exists(g_path) and os.path.exists(l_path):
            g_view = np.nan_to_num(np.load(g_path), nan=0.0)
            l_view = np.nan_to_num(np.load(l_path), nan=0.0)
            
            # Normalization (0 mean, 1 std)
            std_g = np.std(g_view)
            if std_g > 0.0: g_view /= std_g
            
            std_l = np.std(l_view)
            if std_l > 0.0: l_view /= std_l
                
            global_views.append(g_view)
            local_views.append(l_view)
            labels.append(label)
            
    return np.array(global_views), np.array(local_views), np.array(labels)

DATA_DIR = "../data/processed"
MAPPING_CSV = "../data/raw/sample_10stars.csv"

g_data, l_data, y_data = load_curated_dataset(DATA_DIR, MAPPING_CSV)
print(f"Loaded {len(g_data)} valid samples.")

## 2. Prepare Data for Model
Convert labels to categorical (one-hot) for the 4-class classifier, and reshape inputs for the Conv1D layers.

In [ ]:
if len(g_data) > 0:
    num_classes = 4
    y_data_one_hot = tf.keras.utils.to_categorical(y_data, num_classes=num_classes)

    # Shuffle data
    indices = np.random.permutation(len(g_data))
    g_data = g_data[indices].reshape(g_data.shape + (1,))
    l_data = l_data[indices].reshape(l_data.shape + (1,))
    y_data_one_hot = y_data_one_hot[indices]

    print("Global View Shape:", g_data.shape)
    print("Local View Shape:", l_data.shape)
else:
    print("No data found. Ensure preprocessing has been run on sample_10stars.csv.")

## 3. Initialize Advanced AstroNet
The advanced model uses Multi-Head Attention mechanisms. We also compile the model with gradient clipping (`clipnorm=1.0`) and a carefully tuned learning rate (`1e-4`) to prevent NaN loss explosions.

In [ ]:
model_obj = AdvancedAstroNetModel()
model_obj.summary()

## 4. Train the Model
We use Early Stopping and Learning Rate reduction on plateau.

In [ ]:
if len(g_data) > 0:
    os.makedirs("../models", exist_ok=True)
    os.makedirs("../outputs", exist_ok=True)
    
    my_callbacks = [
        callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        callbacks.ModelCheckpoint('../models/astronet_best.keras', monitor='val_accuracy', save_best_only=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5),
    ]

    history = model_obj.model.fit(
        {'global_input': g_data, 'local_input': l_data},
        y_data_one_hot,
        epochs=50,
        batch_size=32,
        validation_split=0.2,
        callbacks=my_callbacks,
        verbose=1
    )
    
    # Save final baseline
    model_obj.save('../models/baseline_model.keras')
    print("Training completed and baseline model saved.")